# Reading From Bronze

- Tables in bronze catelog:
    1. jan2026
    2. feb2026
    3. mar2026
    4. apr2026
    5. may2026
    6. jun2026
    7. airports
    8. weather 

In the silver layer, we will only clean the data for each table. (Merging / data modeling will be done in the Gold Layer)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Data Transformation
Merging all flight data from Jan 2026 - Jun 2026 into one table

Data catelog description from BTS:
| FIELD NAMES | DESCRIPTION |
| ----------- | ----------- |
| FL_DATE | Flight date |
| OP_UNIQUE_CARRIER | Airline Carrier |
| OP_CARRIER_FL_NUM | Flight number |
| ORIGIN | Origin airport IATA code |
| ORIGIN_CITY_NAME | Origin City |
| ORIGIN_STATE_ABR | Origin State |
| DEST | Destination airport IATA code |
| DEST_CITY_NAME | Destination City |
| DEST_STATE_ABR | Destination State |
| CRS_DEP_TIME | Scheduled departure time |
| DEP_TIME | Actual departure time |
| WHEELS OFF | Time aircraft wheels take off |
| WHEELS ON | Time aircraft wheels touch down |
| CRS_ARR_TIME | Scheduled arrival time |
| ARR_TIME | Actual arrival time |
| CANCELLED | 1 indicates yes or true |
| CANCELLATION_CODE | Reason for cancellation |
| DIVERTED | 1 indicates yes or true |
| AIR_TIME | Flight time in minutes |
| DISTANCE | Distance between airports in miles |
| CARRIER_DELAY | Duration in minutes if there is a carrier delay |
| WEATHER_DELAY | Duration in minutes if there is a weather delay |
| NAS_DELAY | Duration in minutes if there is a National Air System Delay |
| SECURITY_DELAY | Duration in minutes if there is a security delay |
| LATE_AIRCRAFT_DELAY | Duration in minutes if there is a delay due to another aircraft |


In [0]:
# Creating dataframe for each month
df_jan = spark.table("bronze_travel.jan2026")
df_feb = spark.table("bronze_travel.feb2026")
df_mar = spark.table("bronze_travel.mar2026")
df_apr = spark.table("bronze_travel.apr2026")
df_may = spark.table("bronze_travel.may2026")
df_jun = spark.table("bronze_travel.jun2026")

In [0]:
jan_count = df_jan.count()
feb_count = df_feb.count()
mar_count = df_mar.count()
apr_count = df_apr.count()
may_count = df_may.count()
jun_count = df_jun.count()
total_count = jan_count + feb_count + mar_count + apr_count + may_count + jun_count

print(f"January: {jan_count}")
print(f"February: {feb_count}")
print(f"March: {mar_count}")
print(f"April: {apr_count}")
print(f"May: {may_count}")
print(f"June: {jun_count}")
print(f"Total: {total_count}")

In [0]:
# Merging all flight dataframes into one
df_flights = (
    df_jan
    .unionByName(df_feb)
    .unionByName(df_mar)
    .unionByName(df_apr)
    .unionByName(df_may)
    .unionByName(df_jun)
)

In [0]:
if total_count != df_flights.count():
    raise ValueError("Dataframe counts do not match")

## Exploring raw data

In [0]:
df_flights.display()

In [0]:
# Check data types

df_flights.dtypes

**Clean Up To Do List:**

- Trim white spaces on every column
- Remove duplicates
- key columns cannot be null: "FL_DATE", "OP_UNIQUE_CARRIER", "OP_CARRIER_FL_NUM", "ORIGIN", "DEST", "CRS_DEP_TIME"
- FL_DATE should be DATE data type
- Time columns should be TIME data type
- Cancelled and diverted columns should be boolean, 0 = false, 1 = true
- Validate categorical/code columns
- Validate numeric values / outliers
- ORIGIN_CITY_NAME should be the city only, remove the ", XY" state
- Check for NULLs
    - Cancelled Flights:
        - Flights that are cancelled should not have any arrival data
        - Flights that are cancelled and have departure time, departure delay, or wheels off time means the trip was aborted mid flight and is to return to an airport (usually origin).
        - If there is a cancellation code, then cancelled column must be true
- Cancellation Codes:
    - A: Carrier Caused - The cancellation was due to circumstances within the airline's control (e.g., maintenance, crew shortages, aircraft cleaning, fueling, or baggage loading). 
    - B: Weather - Significant meteorological conditions (actual or forecasted) that prevented the flight from safely operating (e.g., blizzards, hurricanes, tornadoes).
    - C: National Aviation System (NAS) - Cancellations caused by a broad set of conditions attributable to the aviation system, such as non-extreme weather conditions, heavy airport traffic volume, and air traffic control delays.
    - D: Security - Cancellations resulting from terminal or concourse evacuations, re-boarding due to a security breach, or malfunctioning screening equipment.
- Rename columns

## Trim white spaces from all columns

In [0]:
for column_name, data_type in df_flights.dtypes:
  if data_type == "string":
    df_flights = df_flights.withColumn(column_name, F.trim(F.col(column_name)))

## Drop Duplicates

In [0]:
# columns for unique flight record
key_columns = ["FL_DATE", "OP_UNIQUE_CARRIER", "OP_CARRIER_FL_NUM", "ORIGIN", "DEST", "CRS_DEP_TIME"]

# Find groups with more than one row
dup_groups = (
    df_flights.groupBy(*key_columns)
    .count()
    .filter(F.col("count") > 1)
)

# Number of *extra* rows that dropDuplicates will remove
# (count - 1 per group, summed across all groups)
dup_row_count = (
    dup_groups
    .withColumn("extra_rows", F.col("count") - 1)
    .agg(F.sum("extra_rows"))
    .first()[0]
) or 0  # guard against None if there are no duplicates at all

print(f"Found {dup_row_count} duplicate rows across {dup_groups.count()} groups")

before_count = df_flights.count()

# Drop duplicates, keeping one row per business key
df_flights = df_flights.dropDuplicates(key_columns)

after_count = df_flights.count()
actual_dropped = before_count - after_count

print(f"Before: {before_count}, After: {after_count}, Dropped: {actual_dropped}")

if actual_dropped != dup_row_count:
    raise ValueError(
        f"Expected to drop {dup_row_count} duplicate rows but dropped {actual_dropped}. "
    )

## Check for NULL Values

In [0]:
# Key columns should not be NULL

for column in key_columns:
    null_count = df_flights.select(F.sum(F.col(column).isNull().cast("int")).alias("count")).first()["count"]

    if null_count > 0:
        raise ValueError(f"Column {column} has {null_count} NULL values. Please investigate.")

## Update ORIGIN_CITY_NAME and DEST_CITY_NAME

In [0]:
# Get city name - everything before the last ", XY"
city_pattern = r"^(.*),\s*[A-Za-z]{2}$"

# Finding city names that do not match format "City, ST"

non_matching = df_flights.filter(
    ~F.col("ORIGIN_CITY_NAME").rlike(city_pattern) |
    ~F.col("DEST_CITY_NAME").rlike(city_pattern)
)

non_match_count = non_matching.count()
if non_match_count > 0:
    non_matching.select("ORIGIN_CITY_NAME", "DEST_CITY_NAME").distinct().show()
    raise ValueError(f"{non_match_count} rows don't match expected 'City, ST' pattern")

In [0]:
# Updating city name columns
df_flights = (
    df_flights
        .withColumn("ORIGIN_CITY_NAME", F.regexp_extract(F.col("ORIGIN_CITY_NAME"), city_pattern, 1))
        .withColumn("DEST_CITY_NAME", F.regexp_extract(F.col("DEST_CITY_NAME"), city_pattern, 1))
)

## Cancelled Column

In [0]:
# Check how many CANCELLED values are outside the valid 0/1 domain
invalid_cancelled = df_flights.filter(~F.col("CANCELLED").isin([0, 1]))
invalid_count = invalid_cancelled.count()

print(f"Found {invalid_count} rows with CANCELLED not in (0, 1)")

if invalid_count > 0:
    # Identify unique values in cancelled column
    invalid_cancelled.groupBy("CANCELLED", "CANCELLATION_CODE").count().show()

# For invalid cancelled values, check cancellation code and set to 1 if not null
df_flights = df_flights.withColumn("CANCELLED", F.when(F.col("CANCELLED").isin([0, 1]), F.col("CANCELLED")).when(F.col("CANCELLATION_CODE").isNotNull(), F.lit(1)).otherwise(F.lit(0)))

# Confirm the fix worked — should be 0 now
remaining_invalid = df_flights.filter(~F.col("CANCELLED").isin([0, 1])).count()
print(f"Remaining invalid CANCELLED values after fix: {remaining_invalid}")

if remaining_invalid > 0:
    raise ValueError("CANCELLED still has values outside (0, 1) after correction logic")

In [0]:
# Cancelled flights should not have wheels on, arrival time
odd_cancelled_flights = (
    df_flights.filter((df_flights["CANCELLED"] == 1) & 
              (
                  (df_flights["WHEELS_ON"].isNotNull()) |
                  (df_flights["ARR_TIME"].isNotNull())
                  )
              )
    )
odd_cancelled_flights_count = odd_cancelled_flights.count()
print(f"Found {odd_cancelled_flights_count} cancelled flights with wheels on or arrival time")

In [0]:
# Updating Cancelled column to boolean
df_flights = df_flights.withColumn("CANCELLED", F.when(df_flights["CANCELLED"] == 1, True).otherwise(False))


## Diverted Column

In [0]:
# Check how many DIVERTED values are outside the valid 0/1 domain
invalid_diverted = df_flights.filter(~F.col("DIVERTED").isin([0, 1]))
invalid_count = invalid_diverted.count()

print(f"Found {invalid_count} rows with DIVERTED not in (0, 1)")

if invalid_count > 0:
    # Identify unique values in diverted column
    invalid_diverted.groupBy("DIVERTED").count().show()

# For invalid diverted values, set to 0
df_flights = df_flights.withColumn("DIVERTED", F.when(F.col("DIVERTED").isin([0, 1]), F.col("DIVERTED")).otherwise(F.lit(0)))

# Confirm the fix worked — should be 0 now
remaining_invalid = df_flights.filter(~F.col("DIVERTED").isin([0, 1])).count()
print(f"Remaining invalid DIVERTED values after fix: {remaining_invalid}")

if remaining_invalid > 0:
    raise ValueError("DIVERTED still has values outside (0, 1) after correction")

In [0]:
# Updating Diverted column to boolean
df_flights = df_flights.withColumn("DIVERTED", F.when(df_flights["DIVERTED"] == 1, True).otherwise(False))


## Cancellation Code Column

In [0]:
# Cancellations should only have A, B, C, D, or NULL
invalid_cancellation_code = df_flights.filter(~(F.col("CANCELLATION_CODE").isin(['A', 'B', 'C', 'D']) | F.col("CANCELLATION_CODE").isNull()))
invalid_cancellation_code_count = invalid_cancellation_code.count()

if invalid_cancellation_code_count > 0:
    invalid_cancellation_code.groupBy("CANCELLATION_CODE").count().show()
    raise ValueError("Found invalid cancellation codes. Please investigate.")
else:
    print(f"Found {invalid_cancellation_code_count} invalid cancellation codes.")


In [0]:
# Check if there is a cancellation code then cancelled column should also be true
cancellation_mismatch = df_flights.filter((df_flights["CANCELLATION_CODE"].isNotNull()) & (df_flights["CANCELLED"] == False))
cancellation_mismath_count = cancellation_mismatch.count()

if cancellation_mismath_count > 0:
    print(f"Found {cancellation_mismath_count} rows with a cancellation code but cancelled column is false.")
    raise ValueError("Found rows with a cancellation code but cancelled column is false. Please investigate.")
else:
    print(f"Found {cancellation_mismath_count} rows with a cancellation code but cancelled column is false.")

## Update FL_DATE to DATE Data Type

In [0]:
df_flights = df_flights.withColumn("FL_DATE", F.to_timestamp(F.col("FL_DATE"), "M/d/yyyy h:mm:ss a").cast("date"))

## Update time columns data type
CRS_DEP_TIME, DEP_TIME, WHEELS_OFF, WHEELS_ON, CRS_ARR_TIME, ARR_TIME

In [0]:
# Checking the time value is valid. Cannot be negative or greater than 2400. If there are invalid values, set to NULL.

time_list = ["CRS_DEP_TIME", "DEP_TIME", "WHEELS_OFF", "WHEELS_ON", "CRS_ARR_TIME", "ARR_TIME"]

for column in time_list:

    invalid_time_count = df_flights.filter(
        (df_flights[column] < 0) | 
        (df_flights[column] > 2400)
        ).count()

    if invalid_time_count > 0:
        print(f"Found {invalid_time_count} invalid time values in {column}")

        print(f"Setting invalid value to NULL")

        df_flights = df_flights.withColumn(
            column,
            F.when(
                (F.col(column) < 0) |
                (F.col(column) > 2400),
                None
            ).otherwise(F.col(column))
        )
    else:
        print(f"{column} has {invalid_time_count} invalid time values")

In [0]:
# Left padding with 0 to time columns to 4 digits
df_flights = (
    df_flights
        .withColumn("CRS_DEP_TIME", F.lpad(df_flights["CRS_DEP_TIME"], 4, "0"))
        .withColumn("DEP_TIME", F.lpad(df_flights["DEP_TIME"], 4, "0"))
        .withColumn("WHEELS_OFF", F.lpad(df_flights["WHEELS_OFF"], 4, "0"))
        .withColumn("WHEELS_ON", F.lpad(df_flights["WHEELS_ON"], 4, "0"))
        .withColumn("CRS_ARR_TIME", F.lpad(df_flights["CRS_ARR_TIME"], 4, "0"))
        .withColumn("ARR_TIME", F.lpad(df_flights["ARR_TIME"], 4, "0"))
)


In [0]:
# Handle 2400 (midnight) edge case - convert to 0000
df_flights = (
    df_flights
        .withColumn("CRS_DEP_TIME", F.when(F.col("CRS_DEP_TIME") == "2400", "0000").otherwise(F.col("CRS_DEP_TIME")))
        .withColumn("DEP_TIME", F.when(F.col("DEP_TIME") == "2400", "0000").otherwise(F.col("DEP_TIME")))
        .withColumn("WHEELS_OFF", F.when(F.col("WHEELS_OFF") == "2400", "0000").otherwise(F.col("WHEELS_OFF")))
        .withColumn("WHEELS_ON", F.when(F.col("WHEELS_ON") == "2400", "0000").otherwise(F.col("WHEELS_ON")))
        .withColumn("CRS_ARR_TIME", F.when(F.col("CRS_ARR_TIME") == "2400", "0000").otherwise(F.col("CRS_ARR_TIME")))
        .withColumn("ARR_TIME", F.when(F.col("ARR_TIME") == "2400", "0000").otherwise(F.col("ARR_TIME")))
    
    )

In [0]:
# Converting time columns to TIME data type

df_flights = ( 
          df_flights
            .withColumn("CRS_DEP_TIME", F.to_time(F.col("CRS_DEP_TIME"), F.lit("HHmm")))
            .withColumn("DEP_TIME", F.to_time(F.col("DEP_TIME"), F.lit("HHmm")))
            .withColumn("WHEELS_OFF", F.to_time(F.col("WHEELS_OFF"), F.lit("HHmm")))
            .withColumn("WHEELS_ON", F.to_time(F.col("WHEELS_ON"), F.lit("HHmm")))
            .withColumn("CRS_ARR_TIME", F.to_time(F.col("CRS_ARR_TIME"), F.lit("HHmm")))
            .withColumn("ARR_TIME", F.to_time(F.col("ARR_TIME"), F.lit("HHmm")))
            )

In [0]:
df_flights.dtypes

## Updating data type to INT
AIR_TIME, DISTANCE, CARRIER_DELAY, WEATHER_DELAY, NAS_DELAY, SECURITY_DELAY, LATE_AIRCRAFT_DELAY

In [0]:
# Update double data type columns to int
df_flights = (
    df_flights.withColumn("AIR_TIME", F.col("AIR_TIME").cast("int"))
        .withColumn("DISTANCE", F.col("DISTANCE").cast("int"))
        .withColumn("CARRIER_DELAY", F.col("CARRIER_DELAY").cast("int"))
        .withColumn("WEATHER_DELAY", F.col("WEATHER_DELAY").cast("int"))
        .withColumn("NAS_DELAY", F.col("NAS_DELAY").cast("int"))
        .withColumn("SECURITY_DELAY", F.col("SECURITY_DELAY").cast("int"))
        .withColumn("LATE_AIRCRAFT_DELAY", F.col("LATE_AIRCRAFT_DELAY").cast("int"))
    )

## Renaming Columns

In [0]:
# Rename columns

rename_columns = {
    "FL_DATE": "flight_date",
    "OP_UNIQUE_CARRIER": "airline", 
    "OP_CARRIER_FL_NUM": "carrier_flight_number",
    "ORIGIN": "origin_iata_code",
    "ORIGIN_CITY_NAME": "origin_city",
    "ORIGIN_STATE_ABR": "origin_state",
    "DEST": "destination_iata_code",
    "DEST_CITY_NAME": "destination_city",
    "DEST_STATE_ABR": "destination_state",
    "CRS_DEP_TIME": "scheduled_departure_time",
    "DEP_TIME": "actual_departure_time",
    "WHEELS_OFF": "wheels_off",
    "WHEELS_ON": "wheels_on",
    "CRS_ARR_TIME": "scheduled_arrival_time",
    "ARR_TIME": "actual_arrival_time",
    "CANCELLED": "cancelled",
    "CANCELLATION_CODE": "cancellation_code",
    "DIVERTED": "diverted",
    "AIR_TIME": "air_time",
    "DISTANCE": "distance",
    "CARRIER_DELAY": "carrier_delay",
    "WEATHER_DELAY": "weather_delay",
    "NAS_DELAY": "nas_delay",
    "SECURITY_DELAY": "security_delay",
    "LATE_AIRCRAFT_DELAY": "late_aircraft_delay"
}

In [0]:
# Updating column names
for old_column, new_column in rename_columns.items():
    df_flights = df_flights.withColumnRenamed(old_column, new_column)

# Write Into Silver Table

In [0]:
df_flights.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_travel.flights")

In [0]:
%sql
SELECT *
FROM silver_travel.flights
LIMIT 100